# 03 · Neural Networks, Both Backends

Two things happen here. You build the same network in Keras and in raw PyTorch,
and you run the Keras version under both backends to see for yourself that the
code is genuinely portable — and that the results are not bit-identical.

Then you compare it against the boosted tree from notebook 02, honestly.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
while not (ROOT / "data" / "turbine_readings.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CLEAN = ROOT / "data" / "turbine_clean.csv"
if not CLEAN.exists():
    raise SystemExit("Run Block 1 notebook 02 first — it produces turbine_clean.csv")
print("reading", CLEAN.name)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUM = ["age_years","rotor_speed_rpm","vibration_mm_s","gearbox_temp_c",
       "hours_per_year","vibration_missing"]
CAT = ["site"]
TARGET = "failed_within_90d"

df = pd.read_csv(CLEAN, parse_dates=["install_date"])
df["hours_per_year"] = df["operating_hours"] / df["age_years"].clip(lower=0.1)

X, y = df[NUM + CAT], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

def make_prep():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]), CAT),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"train {X_train.shape}  test {X_test.shape}  failure rate {y.mean():.3f}")

## The Keras version

One environment variable chooses the engine. Set it **before** importing keras —
this is the part people get wrong, because the backend is fixed at import.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"      # must be set before the import below

import keras
print("keras", keras.__version__, "| backend:", keras.backend.backend())

In [ ]:
from sklearn.metrics import average_precision_score, recall_score

prep = make_prep().fit(X_train)
Xtr = prep.transform(X_train); Xte = prep.transform(X_test)
Xtr = np.asarray(Xtr.todense() if hasattr(Xtr, "todense") else Xtr, dtype="float32")
Xte = np.asarray(Xte.todense() if hasattr(Xte, "todense") else Xte, dtype="float32")
ytr = y_train.values.astype("float32"); yte = y_test.values.astype("float32")
print("features after preprocessing:", Xtr.shape[1])

### Normalisation is not optional

`make_prep()` already scales the numeric columns. That is the single most common
reason a network refuses to train — unscaled features make the loss surface
pathological, and no learning rate rescues it.

In [ ]:
def build(seed=0):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([
        keras.layers.Input((Xtr.shape[1],)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
    return m

build().summary()

## First: prove the loop works

Before tuning anything, overfit a single small batch. A correctly wired network
memorises eight examples trivially. If it cannot, the bug is in the wiring, the
data or the loss — and no hyperparameter will find it.

This takes seconds and it is the highest-value ninety seconds in deep learning.

In [ ]:
tiny = build()
h = tiny.fit(Xtr[:8], ytr[:8], epochs=200, verbose=0)
print(f"loss on 8 examples after 200 epochs: {h.history['loss'][-1]:.5f}")
print("✅ loop is wired" if h.history['loss'][-1] < 0.05
      else "❌ cannot overfit 8 rows — stop and debug the loop, not the hyperparameters")

## Now train properly

Class weights, because the rare class must cost more. Early stopping, because a
network will fit noise indefinitely if you let it.

In [ ]:
cw = {0: 1.0, 1: float((ytr == 0).sum() / (ytr == 1).sum())}
print("class weights:", {k: round(v, 2) for k, v in cw.items()})

model = build()
hist = model.fit(Xtr, ytr, validation_split=0.2, epochs=200, batch_size=64,
                 class_weight=cw, verbose=0,
                 callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True,
                                                          monitor="val_loss")])
print(f"stopped after {len(hist.history['loss'])} epochs")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(hist.history["loss"], label="training loss", lw=2)
ax.plot(hist.history["val_loss"], label="validation loss", lw=2)
best = int(np.argmin(hist.history["val_loss"]))
ax.axvline(best, ls="--", color="crimson", label=f"best epoch = {best}")
ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.set_title("Where the two curves separate is where it starts memorising")
ax.legend(); plt.tight_layout()

## The same code, the other backend

Restarting the kernel to change backend is tedious, so run it as a subprocess.
The point is that **the model code is byte-identical** — only the environment
variable differs.

In [ ]:
import subprocess, sys, textwrap, json

SCRIPT = textwrap.dedent('''
    import os, sys, json
    os.environ["KERAS_BACKEND"] = sys.argv[1]
    import numpy as np, keras
    X = np.load(sys.argv[2]); y = np.load(sys.argv[3])
    keras.utils.set_random_seed(0)
    m = keras.Sequential([
        keras.layers.Input((X.shape[1],)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
    h = m.fit(X, y, epochs=5, batch_size=64, verbose=0)
    print(json.dumps({"backend": keras.backend.backend(),
                      "final_loss": round(float(h.history["loss"][-1]), 6)}))
''')

tmp = Path("/tmp"); np.save(tmp/"Xtr.npy", Xtr); np.save(tmp/"ytr.npy", ytr)
(tmp/"backend_test.py").write_text(SCRIPT)

for backend in ["tensorflow", "torch"]:
    r = subprocess.run([sys.executable, str(tmp/"backend_test.py"), backend,
                        str(tmp/"Xtr.npy"), str(tmp/"ytr.npy")],
                       capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("{")]
    print(json.loads(line[-1]) if line else f"{backend}: not available in this environment")

### Read that carefully

Same architecture, same seed, same data — and the final losses differ.

Random number generation and floating-point operation ordering are not identical
across backends. **Reproducibility in deep learning is approximate.** Any test or
evaluation you write must tolerate small differences rather than assert equality,
and any claim that a change "improved" something by a fraction of a percent needs
repeats before you believe it.

## The same network in raw PyTorch

Keras's `fit` is this loop. Writing it once makes that concrete.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(Xtr.shape[1], 32), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([cw[1]]))
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).unsqueeze(1)),
                    batch_size=64, shuffle=True)

for epoch in range(30):
    net.train()
    for xb, yb in loader:
        opt.zero_grad()                       # 1 · clear last step's gradients
        loss = loss_fn(net(xb), yb)           # 2 · forward pass and loss
        loss.backward()                       # 3 · backpropagate
        opt.step()                            # 4 · apply the update
print(f"final batch loss: {loss.item():.4f}")

Four lines, and they are exactly what `fit()` does for you. Everything else
Keras adds — batching, metrics, callbacks, early stopping, progress bars — is
convenience around those four.

## The honest comparison

Against the boosted tree from notebook 02, on the same test set, threshold-free.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = Pipeline([("prep", make_prep()),
               ("clf", HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06,
                                                      max_iter=300, early_stopping=True,
                                                      class_weight="balanced", random_state=0))])
gb.fit(X_train, y_train)

nn_proba = model.predict(Xte, verbose=0).ravel()
gb_proba = gb.predict_proba(X_test)[:, 1]

import time
comparison = pd.DataFrame([
  {"model": "neural network (Keras)",
   "average_precision": round(average_precision_score(yte, nn_proba), 3),
   "recall@0.5": round(recall_score(yte, (nn_proba >= 0.5).astype(int)), 3)},
  {"model": "gradient boosting",
   "average_precision": round(average_precision_score(yte, gb_proba), 3),
   "recall@0.5": round(recall_score(yte, (gb_proba >= 0.5).astype(int)), 3)},
])
comparison

### What to make of that

If the tree matches or beats the network — which on tabular data it usually does —
that is the result, not a failure of your network. The network needed
normalisation, an architecture, a learning rate, an epoch budget, dropout and
early stopping. The tree needed none of those.

**Build the network to learn how. Ship the one that is cheaper to operate.**

---

## Your turn

1. Remove the scaling step from the pipeline and retrain the network. How badly
   does it break? Now do the same to the boosted tree.
2. Set `class_weight` to `None` in both. Which model's recall moves more, and why?
3. Add two more hidden layers. Does validation loss improve, or does the gap
   between the curves just widen sooner?